In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [4]:
#output folders
os.makedirs('/charts', exist_ok=True)
os.makedirs('/website', exist_ok=True)

In [10]:
PALETTE = ['#4361ee','#3a0ca3','#7209b7','#f72585','#4cc9f0',
           '#06d6a0','#ffd166','#ef476f','#118ab2','#073b4c']
sns.set_theme(style='whitegrid', font_scale=1.1)

df_raw = pd.read_csv('Survey_Responses_Combined.csv')
print(f"Raw shape: {df_raw.shape}")

COL = {
    'Timestamp':                                                          'timestamp',
    'Which state would you call your native place?':                      'state',
    'In what region did you primarily grow up?':                          'region',
    'What gender do you identify as?':                                    'gender',
    'What is your sexual orientation?':                                   'orientation',
    'What are your dietary preferences?':                                 'diet',
    "What is your parents' highest educational background?":              'parent_edu',
    'Were you aware of IIT Bhilai before receiving your JEE rank/admission offer?': 'iitbh_awareness',
    'Which degree would you be graduating with?':                         'degree',
    'Which department/program are you graduating from?':                  'dept',
    'Did you pursue any additional coursework outside your core branch (Minor, Honors, extra electives)?': 'extra_course',
    'What is your final CGPA?':                                           'cgpa_bucket',
    'How satisfied are you with your CGPA?':                              'cgpa_sat',
    'Are you interested in pursuing a career in your core branch?':       'core_career',
    'Do you think attending classes is important?':                       'class_imp',
    'How did your typical study schedule look during a semester?':        'study_sched',
    'How did you generally learn new skills/concepts during your time at IIT Bhilai?': 'learn_how',
    'How many F grades have you received?':                               'f_grades',
    'Did you receive any financial aid/scholarship/loan during your degree?': 'financial_aid',
    'In which situations have you NOT used AI tools ever? (select all that apply)': 'ai_not_used',
    'Did you ever witness someone violating the academic honor code?':    'honor_code',
    'How satisfied were you with the quality and variety of electives available?': 'elective_sat',
    'What are your immediate plans after graduation?':                    'post_grad',
    'How well do you think IIT Bhilai has prepared you for life after college?': 'prepared',
    'Where are you heading after passing out? (Abroad/ Mention the city, if in India)': 'destination',
    'How much did your internships help you in making an effective career choice?': 'intern_help',
    '\n\nRate the importance of Interest/alignment with your skillset\n\n': 'imp_interest',
    'Rate the importance of Financial compensation/salary':               'imp_salary',
    'Rate the importance of Place of posting/location':                   'imp_location',
    'Rate the importance of Work culture of the company':                 'imp_culture',
    'Do you think your income will surpass the majority of your batchmates\u2019 income?': 'income_expect',  # U+2019 right quote
    'Did the IIT Bhilai brand/tag help or hinder you during placements/higher studies?': 'brand_help',
    'How frequently did you use dating apps during your stay at IIT Bhilai?': 'dating_apps',
    'How often did you go home during a typical semester':                'go_home',
    'How close were you to your family while at college?':                'family_close',
    'On a scale from 1 to 5, how religious would you say you are?':       'religious',
    'How many people have you dated/been in a relationship with during your time at IIT Bhilai?': 'relationships',
    'Have you made out with anyone?':                                     'makeout',
    'Have you had sex before?':                                           'sex',
    'If you are/were in a relationship, what are your plans?':            'rel_plans',
    'Out of 21 meals per week (3 per day), how many meals did you have in the mess in a typical week?': 'mess_meals',
    "When you didn\u2019t eat in the mess, where did you mostly eat?":         'eat_outside',
    'How did your friendships and social circles change throughout college?': 'friendship_change',
    'How close were you to your roommate?':                               'roommate_close',
    'At what point did you first consume alcohol?':                       'alcohol_start',
    'How often do you currently consume alcohol?':                        'alcohol_freq',
    'At what point did you start smoking (if ever)?':                     'smoke_start',
    'How often do you smoke?':                                            'smoke_freq',
    'How much did you sleep on average every day?':                       'sleep',
    'Which expectation entering college did not turn out as hoped? (select all that apply)': 'unmet_expect',
    'If you could go back in time and say one thing to your fresher self, what would it be? (optional)': 'advice',
    'How did being at a relatively newer IIT affect your overall college experience? (optional)': 'newer_iit',
    'How would you rate the development of campus infrastructure during your four years?': 'infra_rating',
    'Any other thoughts or suggestions about your time at IIT Bhilai? (optional)': 'suggestions',
    'IIT Bhilai in one word (for a word cloud)':                          'one_word',
    'Column 54':                                                          'col54',
}

df = df_raw.rename(columns={k: v for k, v in COL.items() if k in df_raw.columns})
df = df.drop(columns=['col54'], errors='ignore')


Raw shape: (97, 56)


In [11]:

# 3. Cleaning

# 3a. Region: normalise
region_map = {'Tier-3': 'Tier-3 city'}
df['region'] = df['region'].replace(region_map)

# 3b. Diet: normalise non-veg variants
diet_map = {'Non-Vegetarian': 'Always been a Non-vegetarian',
            'Became a Non-vegetarian after IIT Bhilai': 'Became Non-veg in IIT'}
df['diet'] = df['diet'].replace(diet_map)

# 3c. Parent edu: normalise
df['parent_edu'] = df['parent_edu'].replace({'Neither parent graduated': 'Neither parent graduated (School or below)'})

# 3d. Numeric cols: fill NaN with median
num_cols = ['iitbh_awareness','cgpa_sat','class_imp','elective_sat','prepared',
            'intern_help','imp_interest','imp_salary','imp_location','imp_culture',
            'brand_help','family_close','religious','roommate_close','infra_rating']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df[c] = df[c].fillna(df[c].median())

# 3e. F grades: fill NaN with 'Not reported', standardise
df['f_grades'] = df['f_grades'].fillna('Not reported')
df['f_grades'] = df['f_grades'].replace({'2': '1-2', '0': '0'})

# 3f. mess_meals numeric
df['mess_meals'] = pd.to_numeric(df['mess_meals'], errors='coerce').fillna(df['mess_meals'].median())

# 3g. Remove fully blank rows (just in case)
df = df.dropna(how='all')
print(f"Cleaned shape: {df.shape}")


Cleaned shape: (97, 55)


In [12]:

# 4. EDA Summary stats 
eda_summary = {}

# Department distribution
dept_counts = df['dept'].value_counts()
eda_summary['dept'] = dept_counts.to_dict()

# Gender
gender_counts = df['gender'].value_counts()
eda_summary['gender'] = gender_counts.to_dict()

# CGPA bucket
cgpa_order = ['Below 6','6-7','7-8','8-9','9-10']
cgpa_counts = df['cgpa_bucket'].value_counts().reindex(cgpa_order, fill_value=0)
eda_summary['cgpa_bucket'] = cgpa_counts.to_dict()

# Post-grad plans
pg_counts = df['post_grad'].value_counts()
eda_summary['post_grad'] = pg_counts.to_dict()

# Region
reg_counts = df['region'].value_counts()
eda_summary['region'] = reg_counts.to_dict()

# Alcohol start
alc_counts = df['alcohol_start'].value_counts()
eda_summary['alcohol_start'] = alc_counts.to_dict()

# Sleep
sleep_counts = df['sleep'].value_counts()
eda_summary['sleep'] = sleep_counts.to_dict()

# Income expectation
inc_counts = df['income_expect'].value_counts()
eda_summary['income_expect'] = inc_counts.to_dict()

# Numeric averages
numeric_avg = {}
for c in num_cols:
    if c in df.columns:
        numeric_avg[c] = round(float(df[c].mean()), 2)
eda_summary['numeric_avg'] = numeric_avg

print("\n=== EDA Summary ===")
for k, v in eda_summary.items():
    print(f"  {k}: {v}")



=== EDA Summary ===
  dept: {'Electrical Engineering': 20, 'Computer Science and Engineering': 20, 'Data Science and Artificial Intelligence': 16, 'Mechanical Engineering': 16, 'Mechatronics Engineering': 9, 'Chemistry': 5, 'Materials Science and Metallurgical Engineering': 5, 'Physics': 3, 'Mathematics': 1, 'Other': 1, 'B.Tech Electronics and Communication Engineering': 1}
  gender: {'Male': 82, 'Female': 15}
  cgpa_bucket: {'Below 6': 2, '6-7': 14, '7-8': 20, '8-9': 46, '9-10': 15}
  post_grad: {'Job/Placement': 59, 'Higher Studies': 12, 'Competitive Exams': 9, 'Entrepreneurship': 7, 'Not decided yet': 4, 'Figuring it out': 3, 'Startup': 2, 'Other': 1}
  region: {'Tier-2 city': 29, 'Tier-3 city': 21, 'Tier-1 city': 18, 'Rural area': 18, 'Metropolitan city': 11}
  alcohol_start: {'Never': 53, 'Second Year': 13, 'Before IIT Bh': 6, 'Fourth Year': 5, 'First Year': 5, '3rd year': 4, 'Third Year': 3, 'After 3rd year': 2, '2nd year': 2, '1st year': 2, 'Before College': 2}
  sleep: {'5-7 h

In [14]:

# 5. Matplotlib chart exports (for report)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(f'charts/{name}.png', dpi=150, bbox_inches='tight')
    plt.close()

# Chart 1: Department distribution
fig, ax = plt.subplots(figsize=(9, 4))
dept_s = dept_counts.sort_values()
bars = ax.barh(dept_s.index, dept_s.values, color=PALETTE[:len(dept_s)])
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Graduating Department Distribution (n=97)', fontweight='bold')
ax.set_xlabel('Respondents')
save_fig('dept_distribution')

# Chart 2: CGPA distribution
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cgpa_counts.index, cgpa_counts.values, color=PALETTE[:5])
ax.bar_label(bars, padding=2)
ax.set_title('Final CGPA Distribution', fontweight='bold')
ax.set_xlabel('CGPA Range'); ax.set_ylabel('Count')
save_fig('cgpa_distribution')

# Chart 3: Post-grad plans donut
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(pg_counts.values, labels=None,
    autopct='%1.0f%%', colors=PALETTE[:len(pg_counts)], startangle=90,
    wedgeprops={'width': 0.6}, pctdistance=0.8)
ax.legend(wedges, pg_counts.index, loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9)
ax.set_title('Immediate Plans After Graduation', fontweight='bold')
save_fig('post_grad_plans')

# Chart 4: Satisfaction ratings radar-style as bar
sat_cols = ['cgpa_sat','elective_sat','prepared','infra_rating','brand_help']
sat_labels = ['CGPA\nSatisfaction','Elective\nSatisfaction','Prepared for\nLife','Infra\nRating','Brand\nHelp']
sat_means = [df[c].mean() for c in sat_cols]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(sat_labels, sat_means, color=PALETTE[:5])
ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=9)
ax.set_ylim(0, 5.5)
ax.set_title('Average Satisfaction Ratings (1–5 scale)', fontweight='bold')
ax.set_ylabel('Mean Score')
ax.axhline(3, linestyle='--', color='gray', alpha=0.5, label='Neutral (3.0)')
ax.legend()
save_fig('satisfaction_ratings')

# Chart 5: Alcohol consumption start
fig, ax = plt.subplots(figsize=(7, 4))
alc_s = alc_counts
bars = ax.bar(alc_s.index, alc_s.values, color=PALETTE[:len(alc_s)])
ax.bar_label(bars, padding=2)
ax.set_title('When Respondents First Consumed Alcohol', fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=20, ha='right')
save_fig('alcohol_start')

# Chart 6: Sleep distribution
fig, ax = plt.subplots(figsize=(7, 4))
sleep_order = ['Less than 5 hours', '5-7 hours', '7-8 hours', 'More than 8 hours']
sleep_s = df['sleep'].value_counts().reindex(sleep_order, fill_value=0)
bars = ax.bar(sleep_s.index, sleep_s.values, color=PALETTE[2:6])
ax.bar_label(bars, padding=2)
ax.set_title('Average Daily Sleep Duration', fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=10)
save_fig('sleep_distribution')

# Chart 7: Job factor importance (stacked)
job_factors = {
    'Interest / Skillset': df['imp_interest'].value_counts().sort_index(),
    'Financial\nCompensation': df['imp_salary'].dropna().astype(int).value_counts().sort_index(),
    'Location': df['imp_location'].dropna().astype(int).value_counts().sort_index(),
    'Work Culture': df['imp_culture'].dropna().astype(int).value_counts().sort_index(),
}
fig, ax = plt.subplots(figsize=(8, 4))
means = [df[c].mean() for c in ['imp_interest','imp_salary','imp_location','imp_culture']]
labels = ['Interest/\nSkillset','Financial\nComp.','Location','Work\nCulture']
bars = ax.bar(labels, means, color=PALETTE[:4])
ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=9)
ax.set_ylim(0, 5.5)
ax.set_title('Job Factor Importance (avg rating, 1–5)', fontweight='bold')
ax.set_ylabel('Mean Rating')
save_fig('job_factors')

# Chart 8: Core career interest
core_counts = df['core_career'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(core_counts.index, core_counts.values, color=PALETTE[:len(core_counts)])
ax.bar_label(bars, padding=2, fontsize=9)
ax.set_title('Interest in Pursuing Career in Core Branch', fontweight='bold')
ax.set_xlabel('Count')
save_fig('core_career_interest')

# Chart 9: Region of upbringing
fig, ax = plt.subplots(figsize=(7, 4))
reg_s = reg_counts
bars = ax.bar(reg_s.index, reg_s.values, color=PALETTE[:len(reg_s)])
ax.bar_label(bars, padding=2)
ax.set_title('Region Where Respondents Primarily Grew Up', fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=15, ha='right')
save_fig('region_distribution')

# Chart 10: Cross: CGPA bucket vs post_grad (heatmap style)
cross = pd.crosstab(df['cgpa_bucket'].reindex(df.index), df['post_grad'])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(cross.reindex(cgpa_order, fill_value=0), annot=True, fmt='d',
            cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('CGPA Range vs Post-Grad Plans', fontweight='bold')
ax.set_xlabel('Post-Grad Plans'); ax.set_ylabel('CGPA Range')
save_fig('cgpa_vs_postgrad')

print("\n✅ All 10 charts saved")



✅ All 10 charts saved


In [15]:

#6. Export JSON for website 

def safe_dict(d):
    """Convert numpy types to native Python for JSON serialisation."""
    return {str(k): int(v) if isinstance(v, (np.integer,)) else
            float(v) if isinstance(v, (np.floating,)) else v
            for k, v in d.items()}

web_data = {
    "meta": {"n": int(len(df)), "year": "2025-26", "institute": "IIT Bhilai"},

    "dept": safe_dict(dept_counts.to_dict()),
    "gender": safe_dict(gender_counts.to_dict()),
    "region": safe_dict(reg_counts.to_dict()),
    "diet": safe_dict(df['diet'].value_counts().to_dict()),
    "parent_edu": safe_dict(df['parent_edu'].value_counts().to_dict()),

    "cgpa_bucket": {k: int(v) for k, v in cgpa_counts.to_dict().items()},
    "cgpa_sat_avg": round(float(df['cgpa_sat'].mean()), 2),
    "cgpa_sat_dist": safe_dict(df['cgpa_sat'].astype(int).value_counts().sort_index().to_dict()),
    "core_career": safe_dict(df['core_career'].value_counts().to_dict()),
    "class_imp_avg": round(float(df['class_imp'].mean()), 2),
    "study_sched": safe_dict(df['study_sched'].apply(lambda x: x.split(',')[0].strip()).value_counts().to_dict()),
    "elective_sat_avg": round(float(df['elective_sat'].mean()), 2),
    "honor_code": safe_dict(df['honor_code'].value_counts().to_dict()),
    "extra_course": safe_dict(df['extra_course'].value_counts().to_dict()),
    "f_grades": safe_dict(df['f_grades'].value_counts().to_dict()),

    "post_grad": safe_dict(pg_counts.to_dict()),
    "prepared_avg": round(float(df['prepared'].mean()), 2),
    "income_expect": safe_dict(inc_counts.to_dict()),
    "intern_help_avg": round(float(df['intern_help'].mean()), 2),
    "brand_help_avg": round(float(df['brand_help'].mean()), 2),

    "imp_interest_avg": round(float(df['imp_interest'].mean()), 2),
    "imp_salary_avg": round(float(df['imp_salary'].mean()), 2),
    "imp_location_avg": round(float(df['imp_location'].mean()), 2),
    "imp_culture_avg": round(float(df['imp_culture'].mean()), 2),

    "alcohol_start": safe_dict(alc_counts.to_dict()),
    "alcohol_freq": safe_dict(df['alcohol_freq'].value_counts().to_dict()),
    "smoke_start": safe_dict(df['smoke_start'].value_counts().to_dict()),
    "dating_apps": safe_dict(df['dating_apps'].value_counts().to_dict()),
    "relationships": safe_dict(df['relationships'].fillna('Prefer not to say').value_counts().to_dict()),
    "sleep": safe_dict(sleep_s.to_dict()),
    "mess_meals_avg": round(float(df['mess_meals'].mean()), 2),
    "eat_outside": safe_dict(df['eat_outside'].value_counts().to_dict()),
    "go_home": safe_dict(df['go_home'].value_counts().to_dict()),
    "friendship_change": safe_dict(df['friendship_change'].value_counts().to_dict()),

    "infra_rating_avg": round(float(df['infra_rating'].mean()), 2),
    "infra_rating_dist": safe_dict(df['infra_rating'].astype(int).value_counts().sort_index().to_dict()),
    "iitbh_awareness_avg": round(float(df['iitbh_awareness'].mean()), 2),
    "family_close_avg": round(float(df['family_close'].mean()), 2),
    "roommate_close_avg": round(float(df['roommate_close'].mean()), 2),
    "religious_avg": round(float(df['religious'].mean()), 2),

    "orientation": safe_dict(df['orientation'].fillna('Prefer not to say').value_counts().to_dict()),

    # For word cloud: top words
    "one_word": safe_dict(df['one_word'].dropna().str.strip().str.lower().value_counts().head(30).to_dict()),

    # Satisfaction summary for radar chart
    "satisfaction_summary": {
        "CGPA": round(float(df['cgpa_sat'].mean()), 2),
        "Electives": round(float(df['elective_sat'].mean()), 2),
        "Prepared for Life": round(float(df['prepared'].mean()), 2),
        "Infrastructure": round(float(df['infra_rating'].mean()), 2),
        "IIT Brand": round(float(df['brand_help'].mean()), 2),
        "Intern Helpfulness": round(float(df['intern_help'].mean()), 2),
    },

    # CGPA vs post_grad cross table
    "cgpa_vs_postgrad": {
        row: {col: int(val) for col, val in cross.reindex(cgpa_order, fill_value=0).loc[row].to_dict().items()
              if col in cross.columns}
        for row in cgpa_order if row in cross.index
    },
}

json_path = 'website\survey_data.json'
with open(json_path, 'w') as f:
    json.dump(web_data, f, indent=2)
print(f"✅ JSON exported → {json_path}")

# Save cleaned CSV
df.to_csv('website\survey_cleaned.csv', index=False)
print("✅ Cleaned CSV saved → website\survey_cleaned.csv")


✅ JSON exported → website\survey_data.json
✅ Cleaned CSV saved → website\survey_cleaned.csv
